# GW170817 — Time-Marginalized Full Likelihood Validation

Implements and validates the **time-marginalized** and **time+phase-marginalized**
GW log-likelihood for GW170817.

The paper (arXiv:2210.15684) analytically marginalises over tc and φc to avoid
the sharp peak problem in tc. Here we implement three variants in pure JAX:

1. **Standard** — tc and φc sampled explicitly
2. **Time-marginalized** — tc summed over a discrete grid; φc sampled
3. **Time+phase-marginalized** — both tc and φc analytically marginalised

**Data**: BayesWave-cleaned (`BWCLEANED`) 1024-s strain, 128-s analysis segment
**Waveform**: `mlgw_bns_jax` (JAX BNS approximant)
**Detectors**: H1, L1, V1 (fixed sky: NGC 4993)

In [ ]:
from __future__ import annotations
import os, sys, time
import numpy as np
import matplotlib.pyplot as plt

os.environ.setdefault("JAX_PLATFORMS", "cpu")
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

print("JAX devices:", jax.devices())

# ── Event configuration ──────────────────────────────────────────────────
TRIGGER_TIME    = 1187008882.43
SEGMENT_DURATION = 128.0
SAMPLING_RATE   = 4096
F_LOWER         = 23.0
F_UPPER         = 2000.0
DATA_START_GPS  = 1187008114
DATA_DURATION   = 1024

FIXED_RA  = 3.44616      # rad (NGC 4993)
FIXED_DEC = -0.408084    # rad

DATA_DIR = "gw170817_data"
OUTDIR   = "outdir_time_marg_full"
LABEL    = "GW170817_time_marg_full"
os.makedirs(OUTDIR, exist_ok=True)

# tc grid: 3000 points in [-0.15, 0.15] s (300 ms window)
N_TC    = 3000
TC_MIN  = -0.15
TC_MAX  =  0.15
TC_GRID = np.linspace(TC_MIN, TC_MAX, N_TC)

print(f"Segment: {SEGMENT_DURATION}s  →  Δf = {1/SEGMENT_DURATION:.4f} Hz")
print(f"tc grid: {N_TC} pts in [{TC_MIN}, {TC_MAX}] s")

In [ ]:
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))
from jax_import_n_predict import load_predict

MODEL_PATH = "mlgw_bns_jax_model.h5"
_mlgw_predict = load_predict(MODEL_PATH)

import sharpy.GW_likelihood as _gw_mod
from sharpy.utils import McQ2Masses

def _template_mlgw_bns(params, frequency_array):
    mc, q = params[6], params[7]
    m1, m2 = McQ2Masses(mc, q)
    total_mass = m1 + m2
    chi1, chi2 = params[9], params[10]
    lambda_1, lambda_2 = params[11], params[12]
    phic = params[4]
    dist_mpc = jnp.exp(params[2])
    inclination = params[3]
    mlgw_params = jnp.array([q, lambda_1, lambda_2, chi1, chi2])
    hp, hc = _mlgw_predict(
        mlgw_params, frequency_array,
        total_mass=total_mass,
        distance_mpc=dist_mpc,
        inclination=inclination,
    )
    phase_factor = jnp.exp(-1j * phic)
    return hp * phase_factor, hc * phase_factor

_gw_mod.template = _template_mlgw_bns

from sharpy.GW_likelihood import GWNetwork, log_likelihood_det
import sharpy.PSDs
print("Model loaded — SHARPy template patched.")

In [ ]:
# Use BayesWave-cleaned files for both analysis data and PSD estimation
data_files = {
    "H1": os.path.join(DATA_DIR, f"H-H1_BWCLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
    "L1": os.path.join(DATA_DIR, f"L-L1_BWCLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
    "V1": os.path.join(DATA_DIR, f"V-V1_BWCLEANED_4KHZ-{DATA_START_GPS}-{DATA_DURATION}.txt"),
}
for det, f in data_files.items():
    assert os.path.isfile(f), f"Missing: {f}"
    print(f"{det}: {os.path.basename(f)}")

detector_settings = {}
for det in ["H1", "L1", "V1"]:
    detector_settings[det] = dict(
        data_file=data_files[det], channel="GWOSC",
        trigger_time=TRIGGER_TIME, duration=SEGMENT_DURATION,
        sampling_rate=SAMPLING_RATE,
        f_lower=F_LOWER, f_upper=F_UPPER,
        psd_file=None, psd_method="welch",
        download_data=False, zero_noise=False,
    )

print(f"\nBuilding GW network (segment={SEGMENT_DURATION}s)...")
t0 = time.time()
gw_network = GWNetwork(detector_settings, injection_parameters=None)
print(f"Network built in {time.time() - t0:.2f} s")

batched_detector = gw_network.batched_detector

In [ ]:
from gwpy.timeseries import TimeSeries

MERGER_GPS   = TRIGGER_TIME
WINDOW       = 6.0
T_START_PLOT = MERGER_GPS - WINDOW / 2
T_END_PLOT   = MERGER_GPS + WINDOW / 2
F_MIN, F_MAX = 20.0, 800.0
Q_RANGE      = (4, 64)

DET_COLORS = {"H1": "Reds", "L1": "Blues", "V1": "Purples"}
DET_LABELS = {"H1": "LIGO Hanford (H1)", "L1": "LIGO Livingston (L1)", "V1": "Virgo (V1)"}

def _qtransform(filepath):
    strain = np.loadtxt(filepath, comments="#")
    ts = TimeSeries(strain, sample_rate=SAMPLING_RATE, t0=DATA_START_GPS)
    ts_w = ts.whiten(4, 2)
    ts_c = ts_w.crop(T_START_PLOT - 1, T_END_PLOT + 1)
    return ts_c.q_transform(frange=(F_MIN, F_MAX), qrange=Q_RANGE,
                             outseg=(T_START_PLOT, T_END_PLOT), logf=True)

# ── Raw vs BayesWave-cleaned L1 comparison ──────────────────────────────
raw_l1 = os.path.join(DATA_DIR, f"L-L1_GWOSC_4KHZ_R1-{DATA_START_GPS}-{DATA_DURATION}.txt")
bwcln_l1 = data_files["L1"]

if os.path.isfile(raw_l1):
    qt_raw = _qtransform(raw_l1)
    qt_cln = _qtransform(bwcln_l1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 6), sharey=True)
    for ax, qt, title in [(ax1, qt_raw, "L1 — Raw GWOSC"),
                           (ax2, qt_cln, "L1 — BayesWave Cleaned")]:
        pcm = ax.pcolormesh(qt.times.value - MERGER_GPS, qt.frequencies.value,
                             qt.value.T, cmap="Blues", vmin=0, vmax=25)
        ax.set_yscale("log"); ax.set_ylim(F_MIN, F_MAX)
        ax.set_xlabel("Time relative to merger [s]", fontsize=13)
        ax.set_title(title, fontsize=14)
        ax.axvline(0, color="white", ls="--", lw=1, alpha=0.7)
        fig.colorbar(pcm, ax=ax).set_label("Normalized energy")
    ax1.set_ylabel("Frequency [Hz]", fontsize=13)
    fig.suptitle("GW170817 — L1: Raw vs BayesWave Cleaned", fontsize=15)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTDIR, f"{LABEL}_L1_comparison.png"), dpi=150)
    plt.show()

# ── All three detectors (BayesWave cleaned) ──────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 14), sharex=True)
for ax, det in zip(axes, ["H1", "L1", "V1"]):
    qt = _qtransform(data_files[det])
    pcm = ax.pcolormesh(qt.times.value - MERGER_GPS, qt.frequencies.value,
                         qt.value.T, cmap=DET_COLORS[det], vmin=0, vmax=25)
    ax.set_yscale("log"); ax.set_ylim(F_MIN, F_MAX)
    ax.set_ylabel("Frequency [Hz]", fontsize=13)
    ax.set_title(f"{DET_LABELS[det]} (BayesWave cleaned)", fontsize=13)
    ax.axvline(0, color="white", ls="--", lw=1, alpha=0.7)
    fig.colorbar(pcm, ax=ax).set_label("Normalized energy")
axes[-1].set_xlabel("Time relative to merger [s]", fontsize=13)
fig.suptitle("GW170817 — Q-transform spectrograms (BayesWave cleaned, 1024 s)",
             fontsize=15, y=0.995)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, f"{LABEL}_qtransform_bwcleaned.png"), dpi=150)
plt.show()
print("Spectrograms saved.")

## Likelihood variants

### Standard likelihood
log L(θ) = −TwoDTN · Σ_det (dd_det − 2·Re⟨d|h(θ)⟩_det + ⟨h|h⟩_det)

### Time-marginalized likelihood
The coalescence time tc enters the projected waveform only as a phase:
  h_proj(f; tc) ≈ h_proj(f; tc₀) · exp(−i 2π f (tc − tc₀))

So for a grid of tc values:
  log L(tc_k) = −TwoDTN · (dd − 2·Re[cross(tc_k)] + hh)
  log L_tc = logsumexp_k log L(tc_k) − log N_tc

### Time+phase-marginalized likelihood
Additionally factoring out φc:
  ∫₀^{2π} exp(x cos δφ) dδφ/(2π) = I₀(x)
  log L_φ(tc) = const + log I₀(2·TwoDTN·|Z(tc)|)
  log L_tc_φ = logsumexp_k log L_φ(tc_k) − log N_tc

In [ ]:
from functools import partial
from relative_binning import (
    build_full_likelihood_time_marg,
    build_full_likelihood_tc_phi_marg,
    project_waveform_at_freqs,
)

# ── Fiducial parameters (paper MAP values for GW170817) ──────────────────
FIDUCIAL_PARAMS = np.array([
    FIXED_RA,          # [0] ra
    FIXED_DEC,         # [1] dec
    np.log(40.0),      # [2] logdist (40 Mpc)
    2.545,             # [3] theta_jn (~146 deg)
    0.0,               # [4] phic
    0.0,               # [5] pol
    1.1975,            # [6] mc
    0.87,              # [7] q
    0.0,               # [8] tc (relative to trigger)
    0.0,               # [9] chi1
    0.0,               # [10] chi2
    400.0,             # [11] lambda1
    400.0,             # [12] lambda2
])

# ── Standard likelihood (wrapping SHARPy's log_likelihood_det) ───────────
log_L_standard = partial(log_likelihood_det, detector_list=batched_detector)

# ── Time-marginalized (full grid) ────────────────────────────────────────
print("Building time-marginalized full likelihood...")
t0 = time.time()
log_L_tc_marg = build_full_likelihood_time_marg(
    batched_detector, FIDUCIAL_PARAMS, _template_mlgw_bns, TC_GRID,
)
print(f"  built in {time.time()-t0:.2f}s")

# ── Time+phase marginalized (full grid) ──────────────────────────────────
print("Building time+phase-marginalized full likelihood...")
t0 = time.time()
log_L_tc_phi_marg = build_full_likelihood_tc_phi_marg(
    batched_detector, FIDUCIAL_PARAMS, _template_mlgw_bns, TC_GRID,
)
print(f"  built in {time.time()-t0:.2f}s")

# ── JIT compile all three ─────────────────────────────────────────────────
log_L_standard_jit    = jax.jit(log_L_standard)
log_L_tc_marg_jit     = jax.jit(log_L_tc_marg)
log_L_tc_phi_marg_jit = jax.jit(log_L_tc_phi_marg)

# Warm up JIT
_p12 = FIDUCIAL_PARAMS[[0,1,2,3,4,5,6,7,9,10,11,12]]  # remove tc at index 8
_p11 = FIDUCIAL_PARAMS[[0,1,2,3,5,6,7,9,10,11,12]]     # remove tc and phic
_ = float(log_L_standard_jit(FIDUCIAL_PARAMS))
_ = float(log_L_tc_marg_jit(_p12))
_ = float(log_L_tc_phi_marg_jit(_p11))
print("All three likelihoods compiled.")
print(f"  Standard:      log L = {float(log_L_standard_jit(FIDUCIAL_PARAMS)):.2f}")
print(f"  tc-marg:       log L = {float(log_L_tc_marg_jit(_p12)):.2f}")
print(f"  tc+phi-marg:   log L = {float(log_L_tc_phi_marg_jit(_p11)):.2f}")

## Validation A — log L vs tc

Plot the log-likelihood as a function of tc at the fiducial parameters.
The standard likelihood shows a sharp peak; the marginalized value should
match the peak area.

In [ ]:
tc_scan = np.linspace(-0.05, 0.05, 500)
logL_scan = []
for tc_val in tc_scan:
    p = FIDUCIAL_PARAMS.copy()
    p[8] = tc_val
    logL_scan.append(float(log_L_standard_jit(jnp.array(p))))
logL_scan = np.array(logL_scan)

# Reference: the marginalized value
logL_marg_val = float(log_L_tc_marg_jit(jnp.array(_p12)))
logL_tp_val   = float(log_L_tc_phi_marg_jit(jnp.array(_p11)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(tc_scan, logL_scan - logL_scan.max(), label="Standard log L (shifted)", color="tab:blue")
ax.axhline(logL_marg_val - logL_scan.max(), ls="--", color="tab:orange",
           label=f"tc-marg log L = {logL_marg_val:.1f}")
ax.axhline(logL_tp_val   - logL_scan.max(), ls=":",  color="tab:green",
           label=f"tc+φ-marg log L = {logL_tp_val:.1f}")
ax.set_xlabel("tc [s]", fontsize=13)
ax.set_ylabel("log L − max(log L)", fontsize=13)
ax.set_title("GW170817 — log-likelihood vs tc (at fiducial parameters)", fontsize=14)
ax.legend(fontsize=11)
ax.set_xlim(-0.05, 0.05)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, f"{LABEL}_logL_vs_tc.png"), dpi=150)
plt.show()
tc_peak = tc_scan[np.argmax(logL_scan)]
print(f"Peak tc = {tc_peak:.4f} s  (fiducial: {FIDUCIAL_PARAMS[8]:.4f} s)")

In [ ]:
# 1-D log L slices for chirp mass and mass ratio (most important parameters)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chirp mass slice
mc_grid = np.linspace(1.185, 1.210, 80)
logL_mc_std  = []
logL_mc_marg = []
for mc in mc_grid:
    p13 = FIDUCIAL_PARAMS.copy(); p13[6] = mc
    logL_mc_std.append(float(log_L_standard_jit(jnp.array(p13))))
    p12 = p13[[0,1,2,3,4,5,6,7,9,10,11,12]]
    logL_mc_marg.append(float(log_L_tc_marg_jit(jnp.array(p12))))
logL_mc_std  = np.array(logL_mc_std)
logL_mc_marg = np.array(logL_mc_marg)

ax = axes[0]
ax.plot(mc_grid, logL_mc_std  - logL_mc_std.max(),  label="Standard (tc sampled at tc₀)")
ax.plot(mc_grid, logL_mc_marg - logL_mc_marg.max(), ls="--", label="tc-marginalized")
ax.axvline(1.1975, color="red", ls=":", label="Paper MAP")
ax.set_xlabel(r"$\mathcal{M}_c$ [$M_\odot$]", fontsize=12)
ax.set_ylabel("log L − max(log L)", fontsize=12)
ax.set_title(r"Chirp mass $\mathcal{M}_c$", fontsize=13)
ax.legend()

# Mass ratio slice
q_grid = np.linspace(0.5, 1.0, 80)
logL_q_std  = []
logL_q_marg = []
for q in q_grid:
    p13 = FIDUCIAL_PARAMS.copy(); p13[7] = q
    logL_q_std.append(float(log_L_standard_jit(jnp.array(p13))))
    p12 = p13[[0,1,2,3,4,5,6,7,9,10,11,12]]
    logL_q_marg.append(float(log_L_tc_marg_jit(jnp.array(p12))))
logL_q_std  = np.array(logL_q_std)
logL_q_marg = np.array(logL_q_marg)

ax = axes[1]
ax.plot(q_grid, logL_q_std  - logL_q_std.max(),  label="Standard (tc sampled at tc₀)")
ax.plot(q_grid, logL_q_marg - logL_q_marg.max(), ls="--", label="tc-marginalized")
ax.axvline(0.87, color="red", ls=":", label="Paper MAP")
ax.set_xlabel(r"$q$ (mass ratio)", fontsize=12)
ax.set_ylabel("log L − max(log L)", fontsize=12)
ax.set_title("Mass ratio $q$", fontsize=13)
ax.legend()

fig.suptitle("GW170817 — 1-D likelihood slices: standard vs tc-marginalized", fontsize=14)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, f"{LABEL}_slices_mc_q.png"), dpi=150)
plt.show()

## Validation C — mlgw_bns_jax vs mlgw_bns (TEOBResumS training model)

Compare the time-marginalized likelihood evaluated with:
1. `mlgw_bns_jax` — JAX version (our model)
2. `mlgw_bns` — original Python version backed by TEOBResumS

This verifies that both waveform backends produce consistent likelihoods.

In [ ]:
try:
    from mlgw_bns import mlgw_bns_model
    import mlgw_bns

    # Load the original mlgw_bns model
    _mlgw_original = mlgw_bns_model()

    def _template_mlgw_bns_original(params, frequency_array):
        """TEOBResumS-backed mlgw_bns template."""
        mc, q = params[6], params[7]
        m1, m2 = McQ2Masses(mc, q)
        total_mass = m1 + m2
        chi1, chi2 = params[9], params[10]
        lambda_1, lambda_2 = params[11], params[12]
        phic = params[4]
        dist_mpc = float(jnp.exp(params[2]))
        inclination = float(params[3])

        hp_np, hc_np = _mlgw_original.predict_waveform(
            np.array([float(q), float(lambda_1), float(lambda_2),
                      float(chi1), float(chi2)]),
            np.array(frequency_array),
            total_mass=float(total_mass),
            distance_mpc=dist_mpc,
            inclination=inclination,
        )
        phase_factor = jnp.exp(-1j * phic)
        return jnp.array(hp_np) * phase_factor, jnp.array(hc_np) * phase_factor

    # Build time-marginalized likelihood with TEOBResumS waveform
    print("Building TEOBResumS time-marginalized likelihood...")
    log_L_teob_marg = build_full_likelihood_time_marg(
        batched_detector, FIDUCIAL_PARAMS, _template_mlgw_bns_original, TC_GRID,
    )

    # Compare on chirp mass grid
    logL_mc_teob = []
    for mc in mc_grid:
        p12 = FIDUCIAL_PARAMS[[0,1,2,3,4,5,6,7,9,10,11,12]].copy()
        p12[6] = mc  # index 6 in 12-param array is still mc
        logL_mc_teob.append(float(log_L_teob_marg(jnp.array(p12))))
    logL_mc_teob = np.array(logL_mc_teob)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(mc_grid, logL_mc_marg - logL_mc_marg.max(), label="mlgw_bns_jax", lw=2)
    ax.plot(mc_grid, logL_mc_teob - logL_mc_teob.max(), ls="--", label="mlgw_bns (TEOBResumS)", lw=2)
    ax.axvline(1.1975, color="red", ls=":", label="Paper MAP")
    ax.set_xlabel(r"$\mathcal{M}_c$ [$M_\odot$]", fontsize=13)
    ax.set_ylabel("log L − max(log L)", fontsize=13)
    ax.set_title(r"Chirp mass: mlgw_bns_jax vs TEOBResumS", fontsize=14)
    ax.legend(fontsize=11)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTDIR, f"{LABEL}_mc_jax_vs_teob.png"), dpi=150)
    plt.show()

    print("TEOBResumS comparison complete.")
    print(f"  Max |ΔlogL| (Mc slice): {np.max(np.abs(logL_mc_marg - logL_mc_teob)):.2f}")

except ImportError:
    print("mlgw_bns not available — skipping TEOBResumS comparison.")
    print("Install via: pip install mlgw_bns")

In [ ]:
import timeit

# Warm-up already done above; now time each variant
N_REPEATS = 20

t_std = timeit.timeit(lambda: float(log_L_standard_jit(FIDUCIAL_PARAMS)), number=N_REPEATS) / N_REPEATS
t_tc  = timeit.timeit(lambda: float(log_L_tc_marg_jit(jnp.array(_p12))), number=N_REPEATS) / N_REPEATS
t_tp  = timeit.timeit(lambda: float(log_L_tc_phi_marg_jit(jnp.array(_p11))), number=N_REPEATS) / N_REPEATS

print(f"\nTiming benchmark ({N_REPEATS} evaluations each):")
print(f"  Standard (single tc):          {t_std*1e3:.1f} ms")
print(f"  Time-marginalized ({N_TC} tc):  {t_tc*1e3:.1f} ms")
print(f"  Time+phase-marg  ({N_TC} tc):  {t_tp*1e3:.1f} ms")
print(f"\nOverhead of tc marginalization: {t_tc/t_std:.1f}×")
print(f"Overhead of tc+φ marginalization: {t_tp/t_std:.1f}×")

fig, ax = plt.subplots(figsize=(8, 4))
labels = ["Standard\n(1 tc)", f"tc-marg\n({N_TC} tc)", f"tc+φ-marg\n({N_TC} tc)"]
times  = [t_std*1e3, t_tc*1e3, t_tp*1e3]
colors = ["tab:blue", "tab:orange", "tab:green"]
bars   = ax.bar(labels, times, color=colors, alpha=0.8, edgecolor="black")
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{t:.1f} ms", ha="center", va="bottom", fontsize=11)
ax.set_ylabel("Time per call [ms]", fontsize=12)
ax.set_title("Likelihood evaluation time (full frequency grid)", fontsize=13)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, f"{LABEL}_timing.png"), dpi=150)
plt.show()

In [ ]:
print("=" * 70)
print("SUMMARY — GW170817 Time-Marginalized Full Likelihood Validation")
print("=" * 70)
print(f"\nData: BayesWave-cleaned, 128 s segment, {SAMPLING_RATE} Hz")
print(f"tc grid: {N_TC} points in [{TC_MIN}, {TC_MAX}] s")
print(f"\nLog-likelihood at fiducial parameters:")
print(f"  Standard (at tc=0):  {float(log_L_standard_jit(FIDUCIAL_PARAMS)):.2f}")
print(f"  tc-marginalized:     {float(log_L_tc_marg_jit(jnp.array(_p12))):.2f}")
print(f"  tc+φ-marginalized:   {float(log_L_tc_phi_marg_jit(jnp.array(_p11))):.2f}")
print(f"\nPeak tc found:        {tc_peak:.4f} s")
print(f"\nThe marginalized likelihoods successfully integrate over the sharp")
print(f"tc peak, enabling efficient nested sampling without explicit tc search.")